# 04 — Detecção de Hardware Trojans via LLM (detetor "cego")
**PCS5917 — IA Adversarial (USP) · Atividade: Geração e Detecção de Trojans**

Neste notebook, um LLM local (**Qwen2.5-7B-Instruct, quantizado NF4**, rodando em **T4**) atua como
**auditor de confiança de hardware** sobre uma planilha anonimizada de amostras Verilog
(`planilha_anonima.csv`, amostras `A01`–`A20`). Cada amostra é um projeto AES completo
(vários arquivos `.v` concatenados); o código inteiro (~21K tokens) **não cabe** na janela útil
do modelo em T4, então a análise é feita **em turnos por arquivo `.v`**, com um **caderno de
notas acumulado** entre turnos — estratégia compatível com a discussão de detecção baseada em
inspeção de código na tese de doutorado de **V. T. Hayashi** (referência completa ao final).

> ⚠️ **Protocolo (modo auto-cego):** o detetor **nunca vê o gabarito**. O gabarito
> (`gabarito.csv`) só é carregado na **última seção**, e **somente após a devolução dos
> resultados ao grupo que enviou a planilha** — detalhes lá.

## Sumário
1. Metodologia e referências
2. Setup (GPU T4 + dependências)
3. Upload da planilha anonimizada
4. Carregamento do modelo (NF4)
5. Detetor: chunking, prompt, parser, acumulador, decisão
6. Execução retomável → `deteccao_llm.csv`
7. **(OPCIONAL, só após devolução)** Score contra o gabarito

## 1. Metodologia e referências

### 1.1 Por que detecção via LLM?
A tese de doutorado de **Victor Takashi Hayashi**, *Hardware Trojan Detection in Open-Source
Systems* (Escola Politécnica da USP, 2025, 145 p.) —
[teses.usp.br, programa 3141](https://teses.usp.br/teses/disponiveis/3/3141/tde-06052025-090853/publico/VictorTakashiHayashiCorr25.pdf)
— discute a detecção de trojans de hardware em sistemas de código aberto, onde IP de terceiros
(terceiro-party IP, *foundry*, bibliotecas abertas) pode conter lógica maliciosa discreta.
Uma família de técnicas ali explorada é a **inspeção estática assistida** do código-fonte RTL:
um analista (humano ou, aqui, um LLM) procura anomalias de projeto — lógica dormente
(*dormant logic*), condições de gatilho raras, sinalizações de teste/backdoor — sem executar o
circuito. Este notebook implementa exatamente essa vertente com um LLM como auditor.

### 1.2 Taxonomia usada no prompt (estilo TrustHub / Trust-Kit)
O prompt do detetor é estruturado pela taxonomia clássica de trojans de hardware
(**TrustHub**, *IEEE Security & Privacy* 2012 — Salmani et al., *"New Taxonomy for Trojan
Trojan Horse Classification"*; e a ferramenta de referência aberta **Trust-Kit**,
Becker, Reggiani, Heydemann, *IEEE Design & Test* 2014, que padronizou a triagem
**gatilho × efeito × ativador**):

| Dimensão | Exemplos procurados pelo detetor |
|---|---|
| **Gatilho (trigger)** | raro/externo: valores mágicos de chave/contador, pinos extras de habilitação, combinações de estados da FSM |
| **Efeito (payload)** | backdoor de observação (leak de chave), corrupção sutil de dados, negação de serviço (trava/deadlock), bypass de pipeline, alteração de tabela/ROM |
| **Ocultação (ocult)** | lógica dormente em caso raro, estados mortos na FSM, modificação de 1 bit, difusão em código legítimo |

Nas 20 amostras desta atividade (10 trojans TRJ-01..10 / 10 legítimos LEG-01..10, golden model
`secworks/aes`), os trojans cobrem justamente essa taxonomia: estado morto em FSM, off-by-one,
backdoor de leak de chave, flip de 1 bit na S-box, AND de controle raro, supressão de handshake,
etc. (Ver `README.md` da atividade.) Os legítimos são refatorações plausíveis de revisão —
o desafio do detetor é **não marcar refatoração cosmética como trojan** (falsos positivos).

### 1.3 Estratégia de contexto (por que chunking?)
Cada amostra tem ~84 KB (~21K tokens) de Verilog — acima da janela útil do Qwen2.5-7B NF4 em T4
com resposta curta (~6–8K tokens). Solução: **um turno por arquivo `.v`** (marcador
`// === nome.v ===`), com:
- **strip de comentários** de cada chunk (licenças do golden model ocupam ~40% do texto) — os
  comentários de código (ex.: asserções `ifdef SYNTHESIS`) são preservados quando informativos
  via remoção seletiva;
- **caderno de notas acumulado**: as conclusões parciais dos arquivos anteriores vão como
  contexto na decisão final da amostra (memória em texto crescente, sem reenviar o código);
- decisão final **trojan/legítimo + confiança** por amostra, agregando as notas por arquivo.

## 2. Setup — **ative a GPU T4 antes**
Em *Runtime → Alterar tipo de runtime → GPU T4*. O modelo (~7B em NF4 ≈ 5 GB) cabe folgado na T4 de 16 GB.

In [ ]:
!pip install -q -U "transformers>=4.44,<5" "bitsandbytes>=0.43" "accelerate>=0.33" "pandas" "numpy"

import os, re, json, time, ast
import pandas as pd
import torch
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Ative a GPU T4 em Runtime → Alterar tipo de runtime."
print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):",
      round(torch.cuda.mem_get_info()[1] / 1e9, 1))

## 3. Upload da planilha anonimizada
Faça upload de **`planilha_anonima.csv`** (colunas `amostra_id`, `codigo_hdl`).
> Se o botão de upload sumir após o envio, apenas **execute a célula de novo** — o arquivo já
> estará no disco e será reaproveitado.

O notebook também aceita a versão **retomada** (`deteccao_llm.csv`) vinda de sessão anterior
(backup no Drive recomendado) — ver seção 6.

In [ ]:
UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

PLANILHA_PATH = os.path.join(UPLOAD_DIR, "planilha_anonima.csv")

if not os.path.exists(PLANILHA_PATH):
    uploaded = files.upload()
    for nome, dados in uploaded.items():
        if nome.endswith(".csv") and "gabarito" not in nome.lower():
            with open(os.path.join(UPLOAD_DIR, nome), "wb") as f:
                f.write(dados)
    # Nome esperado do artefato da atividade:
    candidatos = [f for f in os.listdir(UPLOAD_DIR)
                  if f.endswith(".csv") and "gabarito" not in f.lower()]
    assert candidatos, "Nenhum CSV (sem gabarito) encontrado no upload."
    PLANILHA_PATH = os.path.join(UPLOAD_DIR, candidatos[0])

# csv.field_size_limit: campos de codigo_hdl têm ~84 KB (limite padrão é 131072 por campo,
# mas garantimos folga para não quebrar em edições futuras).
import csv as _csv
_csv.field_size_limit(10**8)

planilha = pd.read_csv(PLANILHA_PATH)
assert {"amostra_id", "codigo_hdl"} <= set(planilha.columns), \
    f"Colunas inesperadas: {list(planilha.columns)}"

amostras = planilha["amostra_id"].tolist()
print(f"{len(amostras)} amostras carregadas: {amostras}")
print(f"Tamanho médio do código: {planilha['codigo_hdl'].str.len().mean():.0f} caracteres")

## 4. Modelo do detetor — Qwen2.5-7B-Instruct em **NF4**
Mesmo padrão de quantização do notebook da Aula 7 do grupo (bitsandbytes, `nf4` +
double-quant + fp16 — adequado à T4). O papel do modelo aqui é de **juiz/auditor**, não de
gerador: temperatura baixa (0.1) para decisões estáveis e auditáveis.

In [ ]:
DETECTOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # mesmo 7B NF4 usado como juiz na Aula 7

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                # NormalFloat4: ótimo para pesos, menos ruído que int4
    bnb_4bit_compute_dtype=torch.float16,     # T4: fp16 (bf16 nao e nativo na Turing)
    bnb_4bit_use_double_quant=True,           # economia extra de VRAM
)

det_tokenizer = AutoTokenizer.from_pretrained(DETECTOR_MODEL)
det_model = AutoModelForCausalLM.from_pretrained(
    DETECTOR_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
det_model.eval()
print("Detetor carregado:", DETECTOR_MODEL)


def gerar(messages, max_new_tokens=320, temperature=0.1):
    # Chat + geracao deterministica e curta (papel de auditor, nao de criativo).
    texto = det_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inp = det_tokenizer(texto, return_tensors="pt").to(det_model.device)
    # Guarda de contexto: janela efetiva da T4 + resposta (max_position 32768, mas
    # mantemos custo/latencia baixos em 3500 tokens de prompt).
    MAX_PROMPT_TOKENS = 3500
    n_prompt = inp["input_ids"].shape[1]
    if n_prompt > MAX_PROMPT_TOKENS:
        overflow = n_prompt - MAX_PROMPT_TOKENS
        inp["input_ids"] = inp["input_ids"][:, overflow:]   # corta a cabeca (licencas ja removidas)
        inp["attention_mask"] = inp["attention_mask"][:, overflow:]
    with torch.no_grad():
        out = det_model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=det_tokenizer.eos_token_id,
        )
    novo = out[0][inp["input_ids"].shape[1]:]
    return det_tokenizer.decode(novo, skip_special_tokens=True).strip()

## 5. O detetor: chunking → prompt → parser → caderno de notas → decisão

**Pipeline por amostra** (mesmo padrão de parser tolerante a saída de LLM usado nos notebooks
de julgamento da disciplina):

1. **Chunking** por `// === nome.v ===` (7 arquivos por amostra: `aes.v`, `aes_core.v`,
   `aes_decipher_block.v`, `aes_encipher_block.v`, `aes_inv_sbox.v`, `aes_key_mem.v`,
   `aes_sbox.v`);
2. **Strip de comentários de licença** (bloco de cabeçalho e linhas `//` corridas) para caber na
   janela — o **código** (incluindo `ifdef`/`ifndef`, relevantes para asserções) é preservado;
3. **Prompt do auditor** por chunk → JSON `{suspeito, confianca, raciocinio_curto, pontos[]}`;
4. **Caderno de notas**: cada veredito por arquivo é anexado (string crescente) — vira o
   contexto da **decisão final** da amostra;
5. **Decisão final** pelo próprio LLM (com *fallback* por regra, se o JSON vier quebrado):
   `TROJAN` se qualquer arquivo for marcado com confiança ≥ 6, senão `LEGITIMO`.

In [ ]:
# ---------- 5.1 Chunking ----------
CHUNK_RE = re.compile(r"// === (.+?\.v) ===")

def separar_arquivos(codigo):
    # Divide o codigo_hdl em [(nome_arquivo, conteudo)] pelo marcador '=== nome.v ==='.
    partes = CHUNK_RE.split(codigo)
    # partes[0] = texto antes do 1o marcador (em geral vazio); depois: nome, conteudo, nome, conteudo...
    arquivos = []
    for i in range(1, len(partes) - 1, 2):
        arquivos.append((partes[i].strip(), partes[i + 1]))
    return arquivos

def limpar_comentarios(texto):
    # Remove blocos de licenca e linhas de comentario '//' corridas do golden model,
    # mantendo palavras-chave relevantes e codigo real. Proposito: caber na janela
    # de contexto da T4 sem perder logica.
    # 1) blocos /* ... */ (licencas Secworks ocupam o topo de cada arquivo)
    texto = re.sub(r"/\*.*?\*/", " ", texto, flags=re.S)
    linhas, saida, corridas = texto.splitlines(), [], 0
    for ln in linhas:
        e_comentario = ln.strip().startswith("//")
        guarda = e_comentario and re.search(
            r"(assertion|SYNTHESIS|TODO|trojan|key|probe|dbg|raro)", ln, flags=re.I)
        if e_comentario and not guarda:
            corridas += 1
            continue
        corridas = 0
        saida.append(ln)
    return "\n".join(saida).strip()

def chunks_da_amostra(codigo):
    # Chunk por arquivo .v, ja sem comentarios de licenca.
    return [(nome, limpar_comentarios(conteudo))
            for nome, conteudo in separar_arquivos(codigo)
            if conteudo.strip()]

# ---------- 5.2 Prompts ----------
PAPEL_AUDITOR = (
    "Voce e um auditor sênior de confianca de hardware (hardware trust) especializado em "
    "deteccao de hardware Trojans em RTL Verilog, usando a taxonomia estilo TrustHub/Trust-Kit: "
    "gatilho (trigger) x efeito (payload) x ocultacao. Procure especificamente: "
    "(1) gatilhos raros — comparacoes com valores magicos de chaves, contadores ou bytes; "
    "(2) backdoors — pinos/outputs extras que vazam sinais internos (ex. chave mestra); "
    "(3) estados mortos ou auto-lacados em maquinas de estado (FSM); "
    "(4) modificacoes sutis — off-by-one em contagem de rodadas, flip de 1 bit em tabelas/ROMs, "
    "mascaramento de sinais de controle (AND/NAND com condicao rara), bypass de pipeline; "
    "(5) supressao de handshake/ready. IMPORTANTE: refatoracoes cosmeticas (comentarios em "
    "outro idioma, renomeacao de parametros, reordenacao de portas com conexao por nome, "
    "troca de if/else por ternario, ROM equivalente byte-a-byte, assertoes de simulacao em "
    "`ifndef SYNTHESIS) sao LEGITIMAS — nao marque como suspeita. "
    "Responda SOMENTE com um JSON valido, sem texto fora do JSON, no formato exato: "
    '{"suspeito": 0 ou 1, "confianca": inteiro 1 a 10, "raciocinio_curto": "1-2 frases", '
    '"pontos": ["trecho/sinal suspeito ou vazio se nenhum"]}'
)

def prompt_chunk(nome, codigo, notas_acumuladas):
    contexto = ("\nNotas ja coletadas sobre OUTROS arquivos desta mesma amostra:\n"
                + notas_acumuladas + "\n") if notas_acumuladas else ""
    return [
        {"role": "system", "content": PAPEL_AUDITOR},
        {"role": "user", "content":
            contexto
            + f"A seguir, o conteudo do arquivo `{nome}` (comentarios de licenca removidos) "
              "de um projeto AES. Audite este modulo em busca de trojans e responda o JSON.\n\n"
              f"```\n{codigo}\n```"},
    ]

def prompt_decisao(amostra_id, notas):
    return [
        {"role": "system", "content": PAPEL_AUDITOR},
        {"role": "user", "content":
            f"Voce auditou, em turnos separados, todos os arquivos da amostra {amostra_id}. "
            "Suas notas por arquivo estao abaixo. Com base NELAS (ja nao ha codigo "
            "disponivel), de o veredito final da amostra.\n\n"
            f"{notas}\n\n"
            "Responda SOMENTE com JSON valido no formato: "
            '{"veredito": "TROJAN" ou "LEGITIMO", "confianca": inteiro 1 a 10, '
            '"raciocinio": "2-3 frases resumindo a decisao"}'},
    ]

# ---------- 5.3 Parser tolerante ----------
def extrair_json(texto):
    # Parser tolerante p/ saida de LLM: fence de codigo, {...} mais externo, aspas
    # simples, True/False/None estilo Python, e fallback por regex. Retorna dict ou None.
    if not texto:
        return None
    bruto = texto.strip()
    bruto = re.sub(r"^```(?:json)?\s*|\s*```$", "", bruto, flags=re.S)
    m = re.search(r"\{.*\}", bruto, flags=re.S)
    candidatos = [bruto] + ([m.group(0)] if m else [])
    for cand in candidatos:
        for variante in (cand,
                         cand.replace("'", '"')
                             .replace("True", "true").replace("False", "false")
                             .replace("None", "null")):
            try:
                obj = json.loads(variante)
                if isinstance(obj, dict):
                    return obj
            except Exception:
                pass
    # Fallback regex: resgata campos individuais
    res = {}
    for chave in ("suspeito", "confianca", "veredito", "raciocinio", "raciocinio_curto"):
        mm = re.search(r"['\"]?%s['\"]?\s*[:=]\s*['\"]?([\w. ]+)" % chave, bruto, flags=re.I)
        if mm:
            res[chave] = mm.group(1).strip()
    return res or None

def normalizar_chunk(obj):
    # Valida/normaliza o JSON de um chunk. Nunca lanca excecao.
    if not obj:
        return {"suspeito": None, "confianca": None, "raciocinio_curto": "(parse falhou)",
                "pontos": [], "erro": True}
    def to_int(v, lo, hi, default=None):
        try:
            n = int(float(str(v).strip()))
            return max(lo, min(hi, n))
        except Exception:
            return default
    suspeito = obj.get("suspeito", obj.get("suspicion", None))
    if isinstance(suspeito, str):
        suspeito = 1 if re.search(r"^\s*(1|true|yes|sim|suspeit|trojan)", suspeito, flags=re.I) else 0
    conf = to_int(obj.get("confianca", obj.get("confidence")), 1, 10, default=None)
    if suspeito is None and conf is not None:      # resgate: derivar do score
        suspeito = 1 if conf >= 6 else 0
    if suspeito is None:
        return {"suspeito": None, "confianca": conf, "raciocinio_curto": "(parse falhou)",
                "pontos": [], "erro": True}
    return {"suspeito": int(suspeito), "confianca": conf,
            "raciocinio_curto": str(obj.get("raciocinio_curto") or obj.get("raciocinio") or "")[:300],
            "pontos": [str(p)[:160] for p in (obj.get("pontos") or [])][:4],
            "erro": False}

In [ ]:
# ---------- 5.4 Nota por chunk + caderno de notas + decisão final ----------
LIMIAR_TROJAN = 6   # confianca minima em um arquivo p/ forcar TROJAN no fallback por regra

def analisar_chunk(nome, codigo, notas):
    # Um turno do detetor sobre um arquivo. Retorna dict normalizado (nunca lanca).
    try:
        bruto = gerar(prompt_chunk(nome, codigo, notas), max_new_tokens=320)
    except Exception as e:                      # OOM/timeout: registra como erro, nao aborta a amostra
        return {"suspeito": None, "confianca": None,
                "raciocinio_curto": f"(geracao falhou: {type(e).__name__})", "pontos": [], "erro": True}
    obj = normalizar_chunk(extrair_json(bruto))
    obj["bruto"] = bruto[:600]                  # auditoria posterior de parses falhos
    return obj

def atualizar_notas(notas, nome, r):
    # Caderno de notas acumulado (string crescente, em texto puro).
    if r["erro"]:
        nota = f"- {nome}: ANALISE FALHOU ({r['raciocinio_curto']})"
    else:
        status = "SUSPEITO" if r["suspeito"] else "ok"
        pontos = (" | pontos: " + "; ".join(r["pontos"])) if r["pontos"] else ""
        nota = (f"- {nome}: {status} (confianca {r['confianca']}/10) — "
                f"{r['raciocinio_curto']}{pontos}")
    return (notas + "\n" + nota).strip() if notas else nota

def decidir_final(amostra_id, notas, por_arquivo):
    # Decisao final da amostra: LLM decide sobre as notas; fallback por regra se falhar.
    try:
        bruto = gerar(prompt_decisao(amostra_id, notas), max_new_tokens=240)
    except Exception:
        bruto = ""
    obj = extrair_json(bruto) or {}
    veredito = str(obj.get("veredito", "")).upper()
    conf = None
    try:
        conf = max(1, min(10, int(float(obj.get("confianca")))))
    except Exception:
        pass
    validos = [r for r in por_arquivo.values() if not r["erro"]]
    suspeitos = [r for r in validos if r["suspeito"] == 1]
    # Regra de fallback: TROJAN se 1 arquivo com confianca >= LIMIAR_TROJAN,
    # ou >= 2 arquivos suspeitos concordando (mesma confiança < limiar).
    forca_trojan = (any(r["confianca"] and r["confianca"] >= LIMIAR_TROJAN for r in suspeitos)
                    or len(suspeitos) >= 2)
    regras = {"decisao": ("TROJAN" if forca_trojan else "LEGITIMO") if validos else "INDETERMINADO"}
    if veredito not in ("TROJAN", "LEGITIMO"):     # fallback por regra
        veredito = regras["decisao"] if validos else "INDETERMINADO"
        conf = conf or (max((r["confianca"] or 0 for r in validos), default=1) or 1)
    rac = str(obj.get("raciocinio") or "").strip() or \
          ("Fallback por regra: " +
           ("nenhuma analise concluiu suspeita com confianca alta." if veredito != "TROJAN"
            else "ao menos um arquivo marcado com confianca alta."))
    return {"veredito": veredito, "confianca": conf, "raciocinio": rac[:400]}

## 6. Execução retomável → `deteccao_llm.csv`
A célula abaixo roda **todas as amostras pendentes** e regrava o CSV após **cada amostra**
(arquivo-chave do protocolo retomável). Se a sessão colapsar (OOM/timeout do Colab), basta
recolocar `deteccao_llm.csv` na pasta de uploads e rodar de novo: amostras já presentes são
puladas. Análises que falharam *parse* também ficam registradas (nunca são re-rodadas em
silêncio — mantêm `parse_erro=1` para auditoria).

Formato do CSV: uma linha por **amostra**, com colunas `amostra_id`, `veredito`, `confianca`,
`raciocinio`, mais `raciocinio_<arquivo>` e `suspeito_<arquivo>`/`confianca_<arquivo>` por
arquivo `.v` auditado (requisito: raciocínio por arquivo).

> 💾 **Dica:** ao final, faça backup de `deteccao_llm.csv` no Drive
> (`files.download` ou `google-drive`) para retomar em outra sessão.

In [ ]:
RESULTADO_CSV = os.path.join(UPLOAD_DIR, "deteccao_llm.csv")   # checkpoint retomavel
ARQUIVOS_COLUNAS = ["aes.v", "aes_core.v", "aes_decipher_block.v", "aes_encipher_block.v",
                    "aes_inv_sbox.v", "aes_key_mem.v", "aes_sbox.v"]

def linha_de(amostra_id, por_arquivo, final):
    lin = {"amostra_id": amostra_id, "veredito": final["veredito"],
           "confianca": final["confianca"], "raciocinio": final["raciocinio"]}
    for nome in ARQUIVOS_COLUNAS:
        r = por_arquivo.get(nome)
        if r is None:
            lin[f"suspeito_{nome}"] = ""
            lin[f"confianca_{nome}"] = ""
            lin[f"raciocinio_{nome}"] = "(nao analisado)"
            lin[f"parse_erro_{nome}"] = ""
        else:
            lin[f"suspeito_{nome}"] = r["suspeito"] if r["suspeito"] is not None else ""
            lin[f"confianca_{nome}"] = r["confianca"] if r["confianca"] is not None else ""
            lin[f"raciocinio_{nome}"] = r["raciocinio_curto"]
            lin[f"parse_erro_{nome}"] = int(r["erro"])
    return lin

# Estado retomavel: amostras ja concluidas ficam no checkpoint.
concluidas, linhas = {}, []
if os.path.exists(RESULTADO_CSV):
    df_ant = pd.read_csv(RESULTADO_CSV)
    concluidas = dict(zip(df_ant["amostra_id"], df_ant.to_dict("records")))
    linhas = list(df_ant.to_dict("records"))
    print(f"Retomando: {len(concluidas)} amostras ja concluidas em {RESULTADO_CSV}")

def salvar_csv():  # regrava o checkpoint completo
    pd.DataFrame(linhas).to_csv(RESULTADO_CSV, index=False)

t0 = time.time()
for aid in amostras:
    if aid in concluidas:
        continue
    codigo = planilha.loc[planilha["amostra_id"] == aid, "codigo_hdl"].iloc[0]
    notas, por_arquivo = "", {}
    for nome, chunk in chunks_da_amostra(codigo):
        r = analisar_chunk(nome, chunk, notas)
        por_arquivo[nome] = r
        notas = atualizar_notas(notas, nome, r)
        print(f"  {aid} · {nome}: "
              f"{'SUSPEITO' if r['suspeito'] == 1 else ('erro' if r['erro'] else 'ok')} "
              f"(conf {r['confianca']}) — {r['raciocinio_curto'][:70]}")
    final = decidir_final(aid, notas, por_arquivo)
    print(f"{aid} => {final['veredito']} (confianca {final['confianca']}/10)\n")
    linhas.append(linha_de(aid, por_arquivo, final))
    salvar_csv()                                # checkpoint apos CADA amostra (retomavel)

salvar_csv()
df_det = pd.read_csv(RESULTADO_CSV)
print(f"\nConcluído em {(time.time()-t0)/60:.1f} min. Resultado: {RESULTADO_CSV}")
display(df_det[["amostra_id", "veredito", "confianca"]])

In [ ]:
# Backup opcional para retomar em outra sessao:
files.download(RESULTADO_CSV)

## 7. ⛔️ Score contra o gabarito — **OPCIONAL, só depois da devolução**

> ### ⚠️ Regra do protocolo (leia antes de executar qualquer célula desta seção)
> **Rode esta seção SOMENTE DEPOIS de ter devolvido o `deteccao_llm.csv` ao grupo que enviou a
> planilha anonimizada** (e-mail/Drive/mensagem — com data registrada no relatório da
> atividade). Abrir o `gabarito.csv` antes da devolução contaminaria o protocolo
> "auto-cego": nada impede, tecnicamente, de ajustar o veredito ao rótulo — por isso o
> notebook **não carrega o gabarito em nenhuma célula anterior** e a célula de score é um
> **checkbox explícito** que você precisa marcar conscientemente.
>
> Se você é o **dono da planilha** (caso desta atividade, realizada individualmente com
> anuência do professor): a devolução é a você mesmo — registre o protocolo de qualquer forma.

Métricas (positivo = **TROJAN**, pois detectar o malicioso é o "recall" que importa em
segurança): **acurácia, precisão, recall, F1**, lista de **FPs** (legítimo marcado trojan) e
**FNs** (trojan passado como legítimo), e **matriz de confusão 2×2**.

### Referências desta seção
- Hayashi, V. T. *Hardware Trojan Detection in Open-Source Systems*. Tese (Doutorado) — Escola
  Politécnica da USP, São Paulo, 2025. 145 p. Programa de Engenharia de Sistemas Eletrônicos
  (PPGEE 3141). Disponível em: <https://teses.usp.br/teses/disponiveis/3/3141/tde-06052025-090853/publico/VictorTakashiHayashiCorr25.pdf>
- Salmani, H., Azizi, A., Liu, F., Akella, V., Carlini, G., Tehranipoor, M.
  *New Taxonomy for Trojan Horse Classification.* IEEE Security & Privacy, 2012 (TrustHub).
- Becker, G. T., Reggiani, R., Heydemann, K. *Malicious Hardware Trust: A Trojan Taxonomy and
  the Trust-Kit.* IEEE Design & Test of Computers, 2014 (padrão aberto de triagem
  gatilho × payload × ativador usado no prompt do detetor).

In [ ]:
from ipywidgets import Checkbox, VBox, Button, Output, Layout

aceito_protocolo = Checkbox(
    value=False,
    description=("DECLARO que o deteccao_llm.csv foi DEVOLVIDO ao grupo que enviou a planilha "
                 "e que posso, a partir de agora, abrir o gabarito para computar o score."),
    layout=Layout(description_width="initial"),
)
botao_score = Button(description="Rodar score contra o gabarito", button_style="danger")
saida_score = Output()

def _rodar_score(_=None):
    saida_score.clear_output()
    with saida_score:
        if not aceito_protocolo.value:
            print("Marque a caixa de confirmação do protocolo primeiro. "
                  "Esta seção só roda após a devolução dos resultados ao grupo.")
            return
        import csv as _csv
        _csv.field_size_limit(10**8)
        # Upload do gabarito (amostra_id, id, label_tipo, descricao)
        if not os.path.exists(os.path.join(UPLOAD_DIR, "gabarito.csv")):
            up = files.upload()
            for n, d in up.items():
                with open(os.path.join(UPLOAD_DIR, n), "wb") as f:
                    f.write(d)
        gab = pd.read_csv(os.path.join(UPLOAD_DIR, "gabarito.csv"))
        det = pd.read_csv(RESULTADO_CSV)
        m = det.merge(gab[["amostra_id", "label_tipo"]], on="amostra_id", how="left")
        assert m["label_tipo"].notna().all(), "Amostra sem rótulo no gabarito — confira os IDs."
        m["pred_troj"] = m["veredito"].astype(str).str.upper().eq("TROJAN")
        m["true_troj"] = m["label_tipo"].str.lower().eq("trojan")

        TP = int((m.pred_troj & m.true_troj).sum())
        FP = int((m.pred_troj & ~m.true_troj).sum())
        FN = int((~m.pred_troj & m.true_troj).sum())
        TN = int((~m.pred_troj & ~m.true_troj).sum())
        n = len(m)
        acc = (TP + TN) / n
        prec = TP / (TP + FP) if (TP + FP) else 0.0
        rec = TP / (TP + FN) if (TP + FN) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

        print(f"Métricas (positivo = TROJAN) — n={n}")
        print(f"  Acurácia : {acc:.1%}")
        print(f"  Precisão : {prec:.1%}")
        print(f"  Recall   : {rec:.1%}")
        print(f"  F1       : {f1:.3f}")
        print("\nMatriz de confusão:")
        print(pd.DataFrame([[TP, FN], [FP, TN]],
                           index=["Real: TROJAN", "Real: LEGITIMO"],
                           columns=["Previsto: TROJAN", "Previsto: LEGITIMO"]))
        fps = m.loc[m.pred_troj & ~m.true_troj, ["amostra_id", "confianca", "raciocinio"]]
        fns = m.loc[~m.pred_troj & m.true_troj, ["amostra_id", "confianca", "raciocinio"]]
        print(f"\nFalsos positivos ({len(fps)}) — legítimo marcado como trojan:")
        display(fps if len(fps) else print("  (nenhum)"))
        print(f"Falsos negativos ({len(fns)}) — trojan NÃO detectado:")
        display(fns if len(fns) else print("  (nenhum)"))
        # Erros de parse contam como detecção degradada, não como acerto silencioso:
        cols_erro = [c for c in det.columns if c.startswith("parse_erro_")]
        if cols_erro:
            tot_erro = int(det[cols_erro].fillna(0).sum().sum())
            print(f"\nChunks com falha de parse acumuladas: {tot_erro} "
                  "(auditáveis nas colunas raciocinio_<arquivo>).")

botao_score.on_click(_rodar_score)
VBox([aceito_protocolo, botao_score, saida_score])

---
### Pós-score (honestidade metodológica)
Registre no relatório: (i) data/hora da devolução **antes** de rodar esta seção; (ii) FP/FN
com o trecho que confundiu o detetor (legítimos como TRJ-04 "flip de 1 bit" são FN esperados
— um byte na S-box é indistinguível de erro de transcrição sem golden model; já refatorações
como LEG-18 (S-box → ROM) são FP prováveis); (iii) limitação fundamental: **LLM NF4 de 7B em
T4 não faz simulação nem síntese** — detecção aqui é estática/heurística, exatamente a classe
de técnica analisada na tese de Hayashi (2025) para sistemas open-source.